# 06 — Tableau Dashboard Prep

Exports **aggregated, summary-level** tables for the Tableau dashboard — not row-level
data. This matters because `HO_infxn_analysis.csv` is PhysioNet credentialed-access data
under a DUA (see [DATA_ACCESS.md](../DATA_ACCESS.md)); row-level extracts must stay local
and out of any file that might get shared or published alongside the dashboard.

Exports go to `tableau/` (git-ignored for any row-level `.hyper`/`.tde` extracts — the
aggregate CSVs below are small enough to be safe to version, but double-check DUA terms
before making the dashboard itself public).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.data_loading import load_processed

env = load_processed("environmental_mrsa.csv")
pat = load_processed("patient_mrsa.csv")

TABLEAU_DIR = Path("../tableau")
TABLEAU_DIR.mkdir(exist_ok=True)

## Environmental: MRSA acquisition rate by colonization-pressure decile

In [ ]:
env = env.copy()
env["mrsa_cp_decile"] = pd.qcut(env["MRSA_cp"], 10, labels=False, duplicates="drop")

cp_by_decile = (
    env.groupby("mrsa_cp_decile")
    .agg(n=("group_binary", "size"), acquisition_rate=("group_binary", "mean"),
         mean_mrsa_cp=("MRSA_cp", "mean"))
    .reset_index()
)
cp_by_decile.to_csv(TABLEAU_DIR / "env_mrsa_cp_by_decile.csv", index=False)
cp_by_decile

## Patient: MRSA acquisition rate by antibiotic class exposure

In [ ]:
abx_cols = [c for c in pat.columns if c.endswith("_0_60") and c != "any_abx_0_60"]

rows = []
for c in abx_cols:
    exposed = pat[pat[c] > 0]
    unexposed = pat[pat[c] == 0]
    rows.append({
        "antibiotic_class": c.replace("_0_60", ""),
        "n_exposed": len(exposed),
        "acquisition_rate_exposed": exposed["group_binary"].mean(),
        "n_unexposed": len(unexposed),
        "acquisition_rate_unexposed": unexposed["group_binary"].mean(),
    })
abx_summary = pd.DataFrame(rows).sort_values("acquisition_rate_exposed", ascending=False)
abx_summary.to_csv(TABLEAU_DIR / "pat_mrsa_rate_by_abx_class.csv", index=False)
abx_summary

## Model results: odds ratios for the dashboard forest plot

In [ ]:
import pickle

with open("../reports/logit_environmental.pkl", "rb") as f:
    logit_env = pickle.load(f)
with open("../reports/logit_patient_adjusted.pkl", "rb") as f:
    logit_pat_adj = pickle.load(f)


def or_table(model, source_label):
    params = model.params
    conf = model.conf_int()
    conf.columns = ["ci_low", "ci_high"]
    out = np.exp(pd.concat([params, conf], axis=1).rename(columns={0: "coef"}))
    out.columns = ["OR", "ci_low", "ci_high"]
    out["source"] = source_label
    out["variable"] = out.index
    return out.reset_index(drop=True)


or_all = pd.concat([
    or_table(logit_env, "environmental"),
    or_table(logit_pat_adj, "patient"),
], ignore_index=True)
or_all = or_all[or_all["variable"] != "const"]
or_all.to_csv(TABLEAU_DIR / "model_odds_ratios.csv", index=False)
or_all

## Demographic/summary table for dashboard filters

In [ ]:
demo_summary = pd.concat([
    env.assign(analysis="environmental"),
    pat.assign(analysis="patient"),
])[["analysis", "group", "age", "sex", "duration"]]

demo_summary.groupby(["analysis", "group"]).agg(
    n=("age", "size"), mean_age=("age", "mean"), mean_duration=("duration", "mean")
).to_csv(TABLEAU_DIR / "cohort_summary.csv")
pd.read_csv(TABLEAU_DIR / "cohort_summary.csv")

## Dashboard sketch

Planned Tableau views (built from the CSVs above, not raw data):

1. **Acquisition rate vs. MRSA colonization-pressure decile** (line/bar) — environmental arm.
2. **Acquisition rate by antibiotic class, exposed vs. unexposed** (grouped bar) — patient arm.
3. **Forest plot of odds ratios** across both models, side by side, to visually answer the
   "which factor matters more" question.
4. **Cohort summary table** (age, sex, duration by arm/group) as dashboard context/filters.